In [32]:
import pandas as pd
import numpy as np
from dl_client import DatalakeClient

client = DatalakeClient()

In [ ]:
def find_pop_variable(df, file_code, support_file):
    key_pop = []
    for key, value in df.items():
        if 'ADNI1' in value.tolist() or 'ADNI2' in value.tolist() or 'ADNIGO' in value.tolist() or 'ADNI3' in value.tolist() or 'ADNI4' in value.tolist():
            key_pop.append(key)
    
    if len(key_pop)==1:
        pop = key_pop[0]
    elif len(key_pop)>2:
        print(file_code +'\nwe have a problem --> more than 2 key pop')
        print(key_pop)
    elif len(key_pop) ==2:
        if df[key_pop[0]].equals(df[key_pop[1]]):
            pop = key_pop[0]
        else:
            for key in key_pop:
                n = 0
                if not df.groupby('RID')[key].nunique().eq(1).all():
                    n += 1
                    if n>1:
                        print(file_code +'\nwe have a problem --> 2 key pop diverse per sub')
                    pop = key
    else:
        print(file_code +'\nwe have a problem --> NO key pop')
        pop = None
    
    if pop is not None and pop not in support_file[support_file['file_code']==file_code]['variable_code']:
        index = support_file[support_file['file_code']==file_code][0].index()+1
        file_name = support_file[support_file['file_code']==file_code]['file_name']
        new = [file_name, file_code, 'Cohort', None, pop]
        support_file.T.insert(index, index, new)
    return pop


In [23]:
def select_variables(df, file_name, support_file, type='raw/'):
    metadata = client.get_metadata(
        object_name = type+file_name
    )

    file_code = metadata['metadata']['custom']['file_code']
    
    support_file = support_file[support_file['file_code']==file_code]
    lst_variable = [x for x in support_file['variable_code'].unique() if x in list(df.columns)] 
    df_new = df[lst_variable]

    return df_new

In [ ]:
def check_missing_values(df, key, pop_key, stampa=False):  
    n_missing = int(df[key].isna().sum())
    n_tot = int(df.shape[0])
    n_valid = int(n_tot - n_missing)
    
    if pop_key is not None:
        pop = ['ADNI1', 'ADNIGO', 'ADNI2', 'ADNI3', 'ADNI4']
        pop_valid = df[df[key].isna() == False][pop_key].unique().tolist()
        pop_missing = [x for x in pop if x not in pop_valid]
    else:
        pop_valid = ['pop not found']
        pop_missing = ['pop not found']

    if stampa:
        print(f'Missing/total values:        {n_missing}/{n_tot}\nValid/totalvalues:          {n_valid}/{n_tot}')
        print('missing population:  ', pop_missing)
    
    return n_tot, n_valid, n_missing, pop_valid, pop_missing

In [25]:
def check_type_range_variables(df, key):
    n = df[key].first_valid_index()
    tipo = type(df[key][n])
    options = df[key].unique()
    if tipo is not str:
        intervallo = [float(df[key].max()), float(df[key].min())]
    else:
        intervallo = [None]
    
    if len(options) <= 10:
        classes = options
    elif tipo == str:
        classes = options[:5]
    else:
        classes = [None]

    return tipo, intervallo, classes

In [ ]:
def get_varible_info(df, key, file_code, support_file):
    n_tot, n_valid, n_missing, _, pop_missing = check_missing_values(df, key, pop)
    tipo, intervallo, classes = check_type_range_variables(df, key)

    index = support_file.index[(support_file['file_code'] == file_code) & (support_file['variable_code'] == key)][0]

    support_file['type_variable'][index] = tipo
    support_file['classes'][index] = ', '.join(map(str, classes))
    support_file['range'][index] = ', '.join(map(str, intervallo))
    support_file['valid_values'][index] = int(n_valid)
    support_file['missing_values'][index] = int(n_missing)
    support_file['missing_pop'][index] = ', '.join(map(str, pop_missing))
    
    if n_tot/n_valid<= 0.65:
        support_file['del'][index] = True
    else:
        support_file['del'][index] = False

    return support_file

In [27]:
search = client.query_files(
    query={'custom.level' : 'raw'})

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True)

print(zip_files.keys())

dict_keys(['ADNIMERGE_06Jun2025.csv', 'ADSP_PHC_BIOMARKER_06Jun2025.csv', 'BLCHANGE_06Jun2025.csv', 'DXSUM_06Jun2025.csv', 'MMSE_06Jun2025.csv', 'NEUROPATH_06Jun2025.csv', 'PTDEMOG_06Jun2025.csv'])


c:\Users\ChiaraPollicini\anaconda3\envs\aind101\lib\site-packages\dl_client\client.py:490: DtypeWarning: Columns (19,20,21,50,51,104,105,106) have mixed types. Specify dtype option on import or set low_memory=False.
  file_contents[filename] = pd_local.read_csv(io.BytesIO(file_content), delimiter=delimiter)


In [ ]:
support_file = pd.read_excel('ADNI_variables_statistics.xlsx')

for file_name in zip_files.keys():
    df = zip_files[file_name]
    file_code = file_name[:-14]
    pop = find_pop_variable(df, file_code, support_file)

ADNIMERGE COLPROT
ADSP_PHC_BIOMARKER PHASE
BLCHANGE PHASE
DXSUM PHASE
MMSE PHASE
NEUROPATH
we have a problem --> NO key pop
PTDEMOG PHASE


In [ ]:
support_file = pd.read_excel('ADNI_variables_statistics.xlsx')

for file_name in zip_files.keys():
    df = zip_files[file_name]
    file_code = file_name[:-14]
    for key in df.keys():
        get_varible_info(df, key, file_code, support_file)

support_file.to_excel('ADNI_variables_statistics.xlsx', index=False)

## Test su singolo file

In [ ]:
support_file = pd.read_excel('ADNI_variables_statistics.xlsx')
file_code = 'ADNIMERGE'
file_name = 'sjsjdfjfjkonbfd'
pop = 'prova'
index = support_file[support_file['file_code']==file_code].index[1]
print((type(index)))
file_name = support_file[support_file['file_code']==file_code]['file_name']
new = [file_name, file_code, 'Cohorte', None, pop, None, None, None, None, None, None, None]
test = support_file.copy(deep=True)
test.T.insert(index, 'new', value=new)
test

<class 'numpy.int64'>


,file_name,file_code,parameter,population,variable_code,type_variable,classes,range,valid_values,missing_values,missing_pop,del
0,Key ADNI tables merged into one table,ADNIMERGE,ID,"1,GO,2,3",PTID,<class 'str'>,"011_S_0002, 011_S_0003, 022_S_0004, 011_S_0005...",NaN,16421.0,0.0,ADNI4,NaN
1,Key ADNI tables merged into one table,ADNIMERGE,ID,"1,GO,2,3",RID,<class 'numpy.int64'>,NaN,"7125.0, 2.0",16421.0,0.0,ADNI4,NaN
2,Key ADNI tables merged into one table,ADNIMERGE,Cohorte,"1,GO,2,3",COLPROT,<class 'str'>,"ADNI1, ADNI2, ADNIGO, ADNI3",NaN,16421.0,0.0,ADNI4,NaN
3,Key ADNI tables merged into one table,ADNIMERGE,visit,"1,GO,2,3",VISCODE,<class 'str'>,"bl, m06, m12, m24, m18",NaN,16421.0,0.0,ADNI4,NaN
4,Key ADNI tables merged into one table,ADNIMERGE,visit,"1,GO,2,3",EXAMDATE,<class 'str'>,"2005-09-08, 2005-09-12, 2006-03-13, 2006-09-12...",NaN,16421.0,0.0,ADNI4,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
722,Diagnostic Summary,DXSUM,Other,4,DXMPTR5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
723,Diagnostic Summary,DXSUM,Other,4,DXMPTR6,NaN,NaN,NaN,NaN,NaN,NaN,NaN
724,Diagnostic Summary,DXSUM,Other,4,DXPARK,NaN,NaN,NaN,NaN,NaN,NaN,NaN
725,Diagnostic Summary,DXSUM,ID,4,ID,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
file_name = 'ADNIMERGE_06Jun2025.csv'
df = zip_files[file_name]

support_file = pd.read_excel('ADNI_variables_statistics.xlsx')
df_new = select_variables(df, file_name, support_file)

In [ ]:
file_code = 'ADNIMERGE'
for key in df_new.keys():
    get_varible_info(df_new, key, file_code, support_file)

In [ ]:
support_file

In [ ]:
support_file.to_excel('ADNI_variables_statistics.xlsx', index=False)

In [ ]:
df_new['DX'].unique()

In [ ]:
dd_boh = df.apply(lambda x: x.ORIGPROT == x.COLPROT, axis=1)

In [ ]:
dd_boh[dd_boh == False].index

In [ ]:
df_new[df_new['PTRACCAT']=='Black']

In [ ]:
df_new[df_new['AV45'].isna() == False]